# DA213: Synthetic Data Generation - Sampling Effects
**Author:** Rup Narayan Jha (Member C)
**Objective:** Orchestrate Gaussian Copula synthesis and evaluate quality degradation across different sampling methods.

In [10]:
import os
import sys
import warnings
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

for d in ['outputs/synthetic_datasets', 'outputs/evaluation_reports', 'outputs/presentation_figures']:
    os.makedirs(d, exist_ok=True)

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.evaluation.single_table import evaluate_quality

# pulling all our custom methods
from src.sampling_methods import (
    pull_random, custom_seq_sample, bias_cluster_sample, 
    prop_stratified_sample, edge_boundary_sample, 
    logic_condition_sample, skewed_weight_sample
)
from src.quality_metrics import compute_advanced_metrics, generate_evaluation_dashboard

warnings.filterwarnings('ignore')

In [11]:
# helper for generating data
def make_fake_data(real_data, size):
    meta = SingleTableMetadata()
    meta.detect_from_dataframe(real_data)
    
    mod = GaussianCopulaSynthesizer(meta)
    mod.fit(real_data)
    
    return mod.sample(num_rows=size), meta

# getting the basic sdv scores
def get_sdv_scores(real, fake, meta):
    rep = evaluate_quality(real, fake, meta, verbose=False)
    props = rep.get_properties()
    
    return {
        'overall': rep.get_score(),
        'shape': props[props['Property'] == 'Column Shapes']['Score'].values[0],
        'trend': props[props['Property'] == 'Column Pair Trends']['Score'].values[0]
    }

In [12]:
raw = pd.read_csv('data/raw/diabetes_binary_health_indicators_BRFSS2015.csv')

n_total = len(raw)
n_samp = n_total // 100
n_fake = n_total // 10

print(f"loaded {n_total} rows. target sample: {n_samp}. target fake: {n_fake}.")

loaded 253680 rows. target sample: 2536. target fake: 25368.


In [13]:
loops = 10
results = []

for i in range(loops):
    if i % 10 == 0: 
        print(f"running iter {i}...")
    
    # 1. random
    df_rand = pull_random(raw, n=n_samp, seed=i)
    fake_rand, m_rand = make_fake_data(df_rand, n_fake)
    
    res_rand = get_sdv_scores(df_rand, fake_rand, m_rand)
    adv_rand, r1, f1 = compute_advanced_metrics(df_rand, fake_rand, 'BMI')
    res_rand.update({**adv_rand, 'iter': i, 'type': 'random'})
    results.append(res_rand)
    
    # 2. sequence
    df_seq = custom_seq_sample(raw, n=n_samp, seed=i)
    fake_seq, m_seq = make_fake_data(df_seq, n_fake)
    
    res_seq = get_sdv_scores(df_seq, fake_seq, m_seq)
    adv_seq, r2, f2 = compute_advanced_metrics(df_seq, fake_seq, 'BMI')
    res_seq.update({**adv_seq, 'iter': i, 'type': 'sequence'})
    results.append(res_seq)

    # 3. cluster
    df_clu = bias_cluster_sample(raw, col='Education', keep=3, n=n_samp, seed=i)
    fake_clu, m_clu = make_fake_data(df_clu, n_fake)
    
    res_clu = get_sdv_scores(df_clu, fake_clu, m_clu)
    adv_clu, r3, f3 = compute_advanced_metrics(df_clu, fake_clu, 'BMI')
    res_clu.update({**adv_clu, 'iter': i, 'type': 'cluster'})
    results.append(res_clu)

    # 4. stratified
    df_str = prop_stratified_sample(raw, col='Age', n=n_samp, seed=i)
    fake_str, m_str = make_fake_data(df_str, n_fake)
    
    res_str = get_sdv_scores(df_str, fake_str, m_str)
    adv_str, r4, f4 = compute_advanced_metrics(df_str, fake_str, 'BMI')
    res_str.update({**adv_str, 'iter': i, 'type': 'stratified'})
    results.append(res_str)

    # 5. boundary
    df_bnd = edge_boundary_sample(raw, col='BMI', n=n_samp, seed=i)
    fake_bnd, m_bnd = make_fake_data(df_bnd, n_fake)
    
    res_bnd = get_sdv_scores(df_bnd, fake_bnd, m_bnd)
    adv_bnd, r5, f5 = compute_advanced_metrics(df_bnd, fake_bnd, 'BMI')
    res_bnd.update({**adv_bnd, 'iter': i, 'type': 'boundary'})
    results.append(res_bnd)

    # 6. conditional
    df_con = logic_condition_sample(raw, n=n_samp, seed=i)
    fake_con, m_con = make_fake_data(df_con, n_fake)
    
    res_con = get_sdv_scores(df_con, fake_con, m_con)
    adv_con, r6, f6 = compute_advanced_metrics(df_con, fake_con, 'BMI')
    res_con.update({**adv_con, 'iter': i, 'type': 'conditional'})
    results.append(res_con)

    # 7. weighted
    df_wgt = skewed_weight_sample(raw, col='HeartDiseaseorAttack', n=n_samp, seed=i)
    fake_wgt, m_wgt = make_fake_data(df_wgt, n_fake)
    
    res_wgt = get_sdv_scores(df_wgt, fake_wgt, m_wgt)
    adv_wgt, r7, f7 = compute_advanced_metrics(df_wgt, fake_wgt, 'BMI')
    res_wgt.update({**adv_wgt, 'iter': i, 'type': 'weighted'})
    results.append(res_wgt)

    # only save the files at the very end
    if i == loops - 1:
        fake_rand.to_csv('outputs/synthetic_datasets/random_final.csv', index=False)
        generate_evaluation_dashboard(adv_rand, r1, f1, 'BMI', 'outputs/presentation_figures/dash_random.png')
        
        fake_seq.to_csv('outputs/synthetic_datasets/sequence_final.csv', index=False)
        generate_evaluation_dashboard(adv_seq, r2, f2, 'BMI', 'outputs/presentation_figures/dash_sequence.png')
        
        fake_clu.to_csv('outputs/synthetic_datasets/cluster_final.csv', index=False)
        generate_evaluation_dashboard(adv_clu, r3, f3, 'BMI', 'outputs/presentation_figures/dash_cluster.png')
        
        fake_str.to_csv('outputs/synthetic_datasets/stratified_final.csv', index=False)
        generate_evaluation_dashboard(adv_str, r4, f4, 'BMI', 'outputs/presentation_figures/dash_stratified.png')
        
        fake_bnd.to_csv('outputs/synthetic_datasets/boundary_final.csv', index=False)
        generate_evaluation_dashboard(adv_bnd, r5, f5, 'BMI', 'outputs/presentation_figures/dash_boundary.png')
        
        fake_con.to_csv('outputs/synthetic_datasets/conditional_final.csv', index=False)
        generate_evaluation_dashboard(adv_con, r6, f6, 'BMI', 'outputs/presentation_figures/dash_conditional.png')
        
        fake_wgt.to_csv('outputs/synthetic_datasets/weighted_final.csv', index=False)
        generate_evaluation_dashboard(adv_wgt, r7, f7, 'BMI', 'outputs/presentation_figures/dash_weighted.png')

pd.DataFrame(results).to_csv('outputs/evaluation_reports/sampling_metrics_master.csv', index=False)
print("done processing.")

running iter 0...
-> grabbing 2536 random rows out of 253680...
-> running systematic mode (step=100, start=44)
-> cluster sampling... keeping Education categories: [1. 3. 6.]
-> running proportional stratified sampling on Age...
-> grabbing extreme boundaries for BMI...
-> running conditional sampling (HighBP & HighChol)...
-> weighted sampling based on HeartDiseaseorAttack rarity...
-> grabbing 2536 random rows out of 253680...
-> running systematic mode (step=100, start=37)
-> cluster sampling... keeping Education categories: [3. 6. 2.]
-> running proportional stratified sampling on Age...
-> grabbing extreme boundaries for BMI...
-> running conditional sampling (HighBP & HighChol)...
-> weighted sampling based on HeartDiseaseorAttack rarity...
-> grabbing 2536 random rows out of 253680...
-> running systematic mode (step=100, start=40)
-> cluster sampling... keeping Education categories: [2. 6. 5.]
-> running proportional stratified sampling on Age...
-> grabbing extreme boundaries

In [14]:
df_res = pd.read_csv('outputs/evaluation_reports/sampling_metrics_master.csv')

agg = df_res.groupby('type').agg(
    avg_overall=('overall', 'mean'),
    avg_wass=('wasserstein', 'mean'),
    avg_auc=('discriminator_auc', 'mean')
).reset_index()

agg.sort_values(by='avg_overall', ascending=False)

,type,avg_overall,avg_wass,avg_auc
5,stratified,0.917273,1.264629,0.764912
3,random,0.910320,1.468494,0.805403
2,conditional,0.908710,0.670308,0.532571
4,sequence,0.907795,1.534424,0.784053
6,weighted,0.907039,0.993297,0.591239
1,cluster,0.883486,1.088841,0.688796
0,boundary,0.881906,10.339235,0.584320
